# Clase 10 · El mapa de vuelos (gemelo)

**Taller en Ciencia de Datos y Aprendizaje Automático** · Especialización en Estadística · UNAL Manizales · 2026-II
Profesor: Carlos Manuel Orrego Franco

Este cuaderno lo resuelve cada uno en la segunda mitad de la clase. Es el gemelo de *La cifra que falta*: otra vez hay que dar una cifra con su incertidumbre y contarla en una página, ahora con los vuelos de un aeropuerto.

| Ficha | |
|---|---|
| **Objetivo pedagógico** | Calcular una cifra con su intervalo a partir de réplicas, decidir qué hacer con lo que falta en la tabla y entregar el resultado como un tablero que se lee sin correr nada. |
| **Enfoque** | Ciencia de Datos. |
| **Resultados de aprendizaje** | 1 (formular), 2 (preparar datos), 3 (flujo reproducible), 4 (implementar y evaluar), 5 (comunicar) y 6 (reflexión ética, técnica y operativa). |
| **Uso de IA** | El código lo escribe Claude a pedido de ustedes. Deben poder explicar sin ayuda qué decidieron hacer con los meses que no aparecen en la tabla, cómo se calcula el intervalo de la pérdida, por qué no se comparan años calendario incompletos y qué muestra cada panel del tablero. |
| **Entregable** | El cuaderno corrido en su portafolio y el archivo `tablero_vuelos.html` que el cuaderno genera. |

In [ ]:
# Celda de arranque. No hay que cambiarle nada: trae las herramientas del curso.
import pathlib, subprocess, sys
raiz = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "taller" / "datos.py").exists()), None)
if raiz is None:
    subprocess.run(["git", "clone", "-q", "https://github.com/cmorregof/taller-cd-ml-2026-2.git"], check=True)
    raiz = pathlib.Path("taller-cd-ml-2026-2")
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from taller import descargar, procedencia

plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})
print("Herramientas listas.")

In [ ]:
# Los parámetros del cuaderno. Al final se cambia la primera línea y se corre todo de nuevo.
AEROPUERTO = "MZL"        # sigla IATA: MZL es La Nubia, en Manizales
REPLICAS = 500            # réplicas bootstrap para el intervalo

import time

## 1. El enunciado

Imagínense que la Cámara de Comercio y la Alcaldía se van a sentar con las aerolíneas. Quieren llegar con una página que diga qué rutas perdió el aeropuerto, cuántos pasajeros se fueron con cada una y por cuál vale la pena pelear.

1. **Predecir.** Si nada hubiera cambiado, ¿cuántos pasajeros movería hoy cada ruta?
2. **Explicar.** ¿Qué pasó con la ruta que más perdió: se fue una aerolínea, cayó la demanda o cambió el reporte?
3. **Decidir.** ¿Qué ruta perdió más pasajeros y cuál vale la pena recuperar?

**El dato.** La Aeronáutica Civil publica el tráfico aéreo comercial por origen y destino, mes a mes, desde 2020. Cada fila es una empresa, un mes, un trayecto y un tipo de vuelo, con sus pasajeros. Según el diccionario del conjunto, `tipo_vuelo` vale R para regular, A para adicional, C para chárter, N para no regular y T para taxi aéreo. Las coordenadas de los aeropuertos no vienen en ese conjunto: las tomamos de OurAirports, que es un catálogo abierto y no es fuente oficial.

**La cifra.** La pérdida anual de una ruta es la diferencia entre lo que movía en su mejor año y lo que movió en los últimos doce meses:

$$\hat L=12\,(\bar x_{\text{ref}}-\bar x_{\text{rec}}),$$

donde $\bar x_{\text{ref}}$ es el promedio mensual de pasajeros en la ventana de doce meses seguidos con más pasajeros y $\bar x_{\text{rec}}$ es el promedio de los últimos doce meses. Los meses varían, así que $\hat L$ tiene incertidumbre. Se mide con réplicas *bootstrap*: en cada réplica $k$ se sortean con reemplazo doce meses de cada ventana y se recalcula $\hat L_k$. Las réplicas se combinan como en la demostración,

$$\bar L=\frac1m\sum_{k}\hat L_k,\qquad B=\frac{1}{m-1}\sum_k(\hat L_k-\bar L)^2,\qquad \bar L\pm1{,}96\sqrt{\bigl(1+\tfrac1m\bigr)B}.$$

### Criterios de aceptación

1. El cuaderno corre de principio a fin desde cero y funciona con otro aeropuerto cambiando solo la línea `AEROPUERTO`.
2. Hay una tabla mensual completa, con una fila por mes desde el primero hasta el último y una columna por ruta. El cuaderno dice cuántas celdas no venían en el dato y qué se decidió hacer con ellas.
3. Hay una tabla con la pérdida anual de cada ruta, su intervalo de 95 % calculado con al menos 500 réplicas y el porcentaje que representa.
4. El cuaderno muestra con números qué pasa si se comparan años calendario cuando el último año está incompleto.
5. El cuaderno mide qué tan bueno es «lo que movía antes» como pronóstico de «lo que movería hoy», y lo compara con el ancho del intervalo.
6. El cuaderno genera `tablero_vuelos.html`, que se abre en un navegador sin el cuaderno y contiene: un mapa de las rutas, la serie mensual por ruta, la pérdida por ruta con su intervalo y una nota de lectura con la fuente, la fecha del dato y lo que la cifra no dice.
7. Están escritas la decisión en tres frases, la bitácora y las respuestas de riesgos.

### Predicción escrita

- ¿Cuál es la ruta que más pasajeros perdió en el aeropuerto? **Mi apuesta:**
- ¿Qué fracción de las celdas mes por ruta no aparece en el dato? **Mi apuesta:**
- ¿Qué tan ancho será el intervalo de la pérdida de esa ruta, como porcentaje de la pérdida? **Mi apuesta:**

## 2. El dato

In [ ]:
vuelos = descargar("vuelos", aeropuerto=AEROPUERTO)
aeropuertos = descargar("aeropuertos")
print("Filas:", len(vuelos), "| de", f"{vuelos['anio'].min()}-{vuelos.loc[vuelos['anio'] == vuelos['anio'].min(), 'mes'].min():02d}",
      "a", f"{vuelos['anio'].max()}-{vuelos.loc[vuelos['anio'] == vuelos['anio'].max(), 'mes'].max():02d}")
print("Pasajeros por tipo de vuelo:", vuelos.groupby("tipo_vuelo")["pasajeros"].sum().to_dict())
vuelos.head(3)

## 3. Su análisis

De aquí en adelante el cuaderno es suyo. Los títulos siguen los criterios de aceptación.

### 3.1 Limpieza, tabla mensual completa y exploración

In [ ]:
# Limpieza (qué vuelos entran), tabla mensual completa por ruta, decisión sobre lo que falta y exploración.

**Mi lectura** (¿qué se decidió con los meses y las celdas que no venían, y qué otra interpretación tienen?):

### 3.2 La pérdida de cada ruta, con su intervalo

In [ ]:
# Para cada ruta: mejor ventana de 12 meses, últimos 12 meses, pérdida anual, intervalo con réplicas bootstrap
# combinadas y porcentaje de pérdida. Una tabla ordenada de mayor a menor pérdida.

**Mi lectura** (¿qué ruta perdió más?, ¿su intervalo deja por fuera el cero?):

### 3.3 La trampa: comparar años calendario cuando el último está incompleto

In [ ]:
# La caída de la ruta principal calculada de dos maneras: último año calendario contra el mejor año calendario,
# y últimos 12 meses contra la mejor ventana de 12 meses. Mostrar las dos cifras y cuántos meses tiene cada año.

**Mi explicación:**

### 3.4 ¿Qué tan bueno es «lo que movía antes» como pronóstico de «lo que movería hoy»?

La pérdida supone que, si nada hubiera cambiado, la ruta movería hoy lo mismo que en su mejor año. Ese supuesto se puede poner a prueba con la ruta principal: en cada mes del pasado, pronostiquen los doce meses siguientes con tres reglas y midan el error porcentual absoluto, $|\,\text{real}-\text{pronóstico}\,|/\text{real}$.

In [ ]:
# Para la ruta principal, en cada mes del pasado: pronosticar el total de los 12 meses siguientes con al menos
# tres reglas (por ejemplo, los 12 meses anteriores; el promedio de 24; el mejor año visto) y medir el error
# porcentual. Separar los cortes en validación y prueba, en orden de tiempo. Comparar con el intervalo de 3.2.

**Mi lectura** (el intervalo de 3.2 mide cuánto varían los meses; este error mide cuánto falla el supuesto de «si nada hubiera cambiado». ¿Cuál de las dos dudas es mayor, y cuál debe ir en la página?):

### 3.5 Explicar: ¿qué pasó con la ruta que más perdió?

In [ ]:
# Para la ruta que más perdió: pasajeros por año y aerolínea, y último mes con pasajeros de cada aerolínea.

**Mi lectura** (¿qué muestra el dato y qué habría que preguntarle a alguien para saber el porqué?):

### 3.6 El tablero

In [ ]:
# El tablero, con plotly, guardado como tablero_vuelos.html: mapa de rutas, pérdida por ruta con su intervalo,
# serie mensual por ruta, un título que diga el hallazgo con su intervalo y una nota de lectura con la fuente.

Abran `tablero_vuelos.html` en otra pestaña, fuera del cuaderno, y léanlo como lo leería alguien que no estuvo en la clase. ¿Entiende la cifra? ¿Sabe qué no dice?

### La decisión en tres frases

Para quien se va a sentar con las aerolíneas.

1. **Recomendación:**
2. **Evidencia:**
3. **Riesgo:**

### 3.7 Bitácora, riesgos y variante

**Bitácora de decisiones.** Al menos cinco filas.

| Decisión | Alternativa descartada | Por qué |
|---|---|---|
| | | |

**Riesgos y mantenimiento.** Respondan en dos o tres líneas cada una.

- Un mes que no aparece en el dato se contó como cero. ¿En qué caso eso es falso, y cómo lo comprobarían con la Aeronáutica Civil?
- La cifra es de pasajeros que volaron. ¿Qué diferencia hay entre «pasajeros perdidos» y «demanda perdida»? ¿A dónde pudieron irse esos pasajeros?
- El mejor año de una ruta puede ser un año raro. ¿Qué le pasa a la pérdida si la referencia fue un pico que no se iba a repetir?
- El conjunto se actualiza. ¿Qué parte del tablero queda vieja primero y cómo sabría el lector de qué fecha es?

**Cambien una línea.** Pongan `AEROPUERTO = "PEI"` (Matecaña, en Pereira), corran todo y llenen la tabla.

| | MZL | PEI |
|---|---|---|
| Rutas con más de mil pasajeros en su mejor año | | |
| Ruta que más perdió | | |
| Pérdida anual de esa ruta, con intervalo | | |
| Celdas mes por ruta que no venían en el dato (%) | | |
| Error mediano en prueba de «el mejor año visto» (%) | | |

¿Qué cuentan las dos tablas juntas que ninguna cuenta sola? **Mi respuesta:**

### Procedencia y entrega

In [ ]:
{nombre: ficha for nombre, ficha in procedencia().items() if nombre in ("vuelos", "aeropuertos")}

**Para entregar:** el cuaderno corrido y `tablero_vuelos.html` en su portafolio, carpeta `clase10/`. En Colab el tablero queda en el panel de archivos: descárguenlo y súbanlo junto al cuaderno.